# 🍃 TobaccoGuard AI - Training & Evaluasi Model di Google Colab

Notebook ini digunakan untuk melatih dan membandingkan 2 arsitektur Deep Learning:
1. **MobileNetV2 (Modern CNN + Transfer Learning)**: Model utama hemat daya & akurasi tinggi.
2. **AlexNet (Klasik CNN dari Awal)**: Model pembanding empiris untuk analisis komparasi.

---
### 📌 Alur Singkat:
1. **Runtime GPU**: Pastikan menu **Runtime** > **Change runtime type** sudah diset ke **T4 GPU**.
2. **Upload `ai.zip`**: Jalankan Sel 2 untuk mengunggah dataset dan kode.
3. **Jalankan Training**: Jalankan Sel MobileNetV2 dan Sel AlexNet.
4. **Lihat Perbandingan**: Sel 6 akan menampilkan grafik dan tabel perbandingan akurasi secara otomatis.

## 1. Cek GPU NVIDIA

In [ ]:
!nvidia-smi

## 2. Upload File `ai.zip`
Klik tombol **Choose Files** dan pilih file `ai.zip` dari folder project Anda.

In [ ]:
import os
from google.colab import files

if not os.path.exists('/content/ai'):
    print("Silakan pilih dan upload file 'ai.zip':")
    uploaded = files.upload()
    !unzip -q ai.zip -d /content/
    print("\nEkstraksi selesai! Folder /content/ai siap.")
else:
    print("Folder /content/ai sudah ada.")

## 3. Install Dependensi

In [ ]:
%cd /content/ai
!pip install -q -r requirements.txt

## 4. Run Training & Akurasi: MobileNetV2 (Transfer Learning)
Menjalankan arsitektur MobileNetV2 pretrained ImageNet dengan 2 tahap pelatihan:
- Tahap 1: Feature Extraction (8 Epoch)
- Tahap 2: Fine-Tuning 30 layer teratas (6 Epoch)

In [ ]:
%cd /content/ai
!python train.py

## 5. Run Training & Akurasi: AlexNet (CNN Klasik)
Menjalankan arsitektur AlexNet klasik (5 Convolutional Layers + 3 Dense Layers) dilatih dari nol (*from scratch*) sebagai model pembanding.

In [ ]:
%cd /content/ai
!python train_alexnet.py

## 6. Tampilkan Tabel & Grafik Perbandingan Akurasi (MobileNetV2 vs AlexNet)
Sel ini akan membaca metadata hasil kedua model, mencetak tabel perbandingan akurasi, dan menghasilkan grafik perbandingan.

In [ ]:
%cd /content/ai
# Buat grafik perbandingan
!python generate_comparison_charts.py

# Tampilkan Ringkasan Tabel Akurasi
import json
import pandas as pd
from IPython.display import Image, display

with open('/content/ai/models/model_metadata.json') as f:
    mb = json.load(f)
with open('/content/ai/models/alexnet_metadata.json') as f:
    al = json.load(f)

summary_data = {
    "Metrik": [
        "Arsitektur",
        "Metode Pelatihan",
        "Akurasi Validasi",
        "Validation Loss",
        "F1-Score Filler (Low)",
        "F1-Score Binder (Med)",
        "F1-Score Wrapper (High)",
        "Total Parameter"
    ],
    "MobileNetV2": [
        "MobileNetV2 (Modern CNN)",
        "Transfer Learning & Fine-Tuning",
        f"{mb['validation_accuracy']*100:.2f}%",
        f"{mb['validation_loss']:.4f}",
        f"{mb['classification_report']['filler']['f1-score']*100:.1f}%",
        f"{mb['classification_report']['binder']['f1-score']*100:.1f}%",
        f"{mb['classification_report']['wrapper']['f1-score']*100:.1f}%",
        f"{mb['total_parameters']:,}"
    ],
    "AlexNet": [
        "AlexNet (Klasik CNN)",
        "Training from Scratch",
        f"{al['validation_accuracy']*100:.2f}%",
        f"{al['validation_loss']:.4f}",
        f"{al['classification_report']['filler']['f1-score']*100:.1f}%",
        f"{al['classification_report']['binder']['f1-score']*100:.1f}%",
        f"{al['classification_report']['wrapper']['f1-score']*100:.1f}%",
        f"{al['total_parameters']:,}"
    ]
}

df = pd.DataFrame(summary_data)
print("\n" + "="*60)
print("    HASIL KOMPARASI AKURASI: MOBILENETV2 VS ALEXNET")
print("="*60)
display(df)

print("\n--- Grafik Perbandingan Akurasi & Loss ---")
display(Image('/content/ai/reports/perbandingan_akurasi_loss.png', width=750))

print("\n--- Grafik Perbandingan F1-Score Tiap Mutu ---")
display(Image('/content/ai/reports/perbandingan_f1_score.png', width=750))


## 7. Download Semua Model & Grafik ke Laptop Anda

In [ ]:
from google.colab import files
!zip -r /content/tobacco_trained_models.zip /content/ai/models /content/ai/reports
files.download('/content/tobacco_trained_models.zip')